In [20]:
import sklearn
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [21]:

# Load the dataset
coffee_data = pd.read_csv(r"..\data\reduced_data_bin.csv")

# Create the target column
coffee_data['target'] = coffee_data['caffeine_class'].map(lambda x: 0 if x == 'Absent' else 1)

# Store the column names before scaling
column_names = coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']).columns

# Scale the features
scaler = StandardScaler()
X_scaled_std = scaler.fit_transform(coffee_data.drop(columns=['caffeine_class', 'caffeine_percent', 'target']))

# Define X and y
y = coffee_data['target']
X = X_scaled_std.copy()

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Get feature importances
importances = model.feature_importances_

# Use the stored column names to map the importances back to the feature names
important_features = sorted(zip(column_names, importances), key=lambda x: x[1], reverse=True)
feature_importance_dict = {feature: importance for feature, importance in zip(column_names, importances)}

# Display the most important features
for feature, importance in important_features:
    print(f"Feature: {feature}, Importance: {importance:.4f}")


Feature: clim_34_prec10_oct, Importance: 0.0726
Feature: env_75_geo_10, Importance: 0.0676
Feature: clim_14_tmax2_feb, Importance: 0.0625
Feature: clim_25_prec1_jan, Importance: 0.0538
Feature: clim_1_tmin1_jan, Importance: 0.0519
Feature: clim_40_temp_season, Importance: 0.0491
Feature: clim_27_prec3_mar, Importance: 0.0484
Feature: env_76_soi_10, Importance: 0.0457
Feature: clim_13_tmax1_jan, Importance: 0.0442
Feature: clim_26_prec2_feb, Importance: 0.0435
Feature: clim_35_prec11_nov, Importance: 0.0405
Feature: clim_38_mean_diurn_range, Importance: 0.0402
Feature: clim_39_isotherm, Importance: 0.0382
Feature: clim_28_prec4_apr, Importance: 0.0382
Feature: env_74_solrad, Importance: 0.0361
Feature: clim_69_cwd_annual, Importance: 0.0328
Feature: env_78_wat_2, Importance: 0.0322
Feature: env_73_asp, Importance: 0.0290
Feature: env_72_slo, Importance: 0.0270
Feature: env_79_forcov, Importance: 0.0191
Feature: env_76_soi_9, Importance: 0.0135
Feature: env_76_soi_21, Importance: 0.0132


In [22]:
importance_threshold = 0.001

# Identify features to remove
features_to_remove = [feature for feature, importance in feature_importance_dict.items() if importance < importance_threshold]

# Drop low-importance features from the dataset
#X_reduced = X.drop(columns=features_to_remove)
indices_to_remove = [list(column_names).index(feature) for feature in features_to_remove]

# Remove the low-importance features from the scaled NumPy array
X_reduced = np.delete(X, indices_to_remove, axis=1)

# Check the shape of the reduced feature matrix to ensure the columns have been removed
print(f"Original shape: {X.shape}")
print(f"Reduced shape: {X_reduced.shape}")
# Print out the features that are removed
print(f"Features removed: {features_to_remove}")

Original shape: (526, 78)
Reduced shape: (526, 46)
Features removed: ['env_75_geo_5', 'env_75_geo_11', 'env_76_soi_7', 'env_76_soi_23', 'env_76_soi_20', 'env_76_soi_19', 'env_76_soi_3', 'env_76_soi_15', 'env_76_soi_12', 'env_76_soi_22', 'env_76_soi_17', 'env_76_soi_8', 'env_76_soi_4', 'env_77_veg_1', 'env_77_veg_2', 'env_77_veg_3', 'env_77_veg_7', 'env_77_veg_9', 'env_77_veg_11', 'env_77_veg_12', 'env_77_veg_13', 'env_77_veg_22', 'env_78_wat_12', 'env_78_wat_11', 'env_78_wat_8', 'env_78_wat_20', 'env_78_wat_13', 'env_78_wat_22', 'env_78_wat_17', 'env_78_wat_1.1', 'env_78_wat_18', 'env_78_wat_15']


In [27]:
X_reduced_df = pd.DataFrame(X_reduced, columns=[col for col in column_names if col not in features_to_remove])
X_reduced_df['caffeine_class'] = coffee_data['caffeine_class']
X_reduced_df['caffeine_percent'] = coffee_data['caffeine_percent']
X_reduced_df.to_csv(r'../data/reduced_for_training.csv', index=False)


In [23]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define X and y (this assumes your data is already prepared as in your example)
X = X_reduced.copy()  # Your scaled data
y = coffee_data['target']

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define the Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Define the scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'roc_auc': make_scorer(roc_auc_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score)
}

# Perform cross-validation
cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)

# Print cross-validation results
print("Cross-validation results (mean values):")
print(f"Accuracy: {cv_results['test_accuracy'].mean():.4f}")
print(f"ROC-AUC: {cv_results['test_roc_auc'].mean():.4f}")
print(f"Precision: {cv_results['test_precision'].mean():.4f}")
print(f"Recall: {cv_results['test_recall'].mean():.4f}")
print(f"F1-Score: {cv_results['test_f1'].mean():.4f}")


Cross-validation results (mean values):
Accuracy: 0.9348
ROC-AUC: 0.9087
Precision: 0.9200
Recall: 0.8514
F1-Score: 0.8810
